___
<a href='https://cafe.naver.com/jmhonglab'><p style="text-align:center;"><img src='https://lh3.googleusercontent.com/lY3ySXooSmwsq5r-mRi7uiypbo0Vez6pmNoQxMFhl9fmZJkRHu5lO2vo7se_0YOzgmDyJif9fi4_z0o3ZFdwd8NVSWG6Ea80uWaf3pOHpR4GHGDV7kaFeuHR3yAjIJjDgfXMxsvw=w2400'  class="center" width="50%" height="50%"/></p></a>
___
<center><em>Content Copyright by HongLab, Inc.</em></center>

# 가상 광고판 만들기

[Pub Crowd Reaction meme](https://www.youtube.com/watch?v=755i2tTgexo)  

- Dest 영상은 [유튜브 영상(STL Ballpark Village reaction)](https://www.youtube.com/watch?v=IOC-MoJwxTU)에서 일부를 잘라낸 후 안정화(stabilization)
- AR에도 비슷한 원리가 적용됩니다.

절차 (이미지 파일 단위로 작업해봅시다)
1. 원본(source) 영상과 대상(destination) 영상을 프레임 단위의 이미지 시퀀스로 저장
2. 합성된 이미지 시퀀스 생성
3. 이미지 시퀀스를 다시 동영상으로 묶기
4. 오디오 합성

### FFMpeg

[ffmpeg-python](https://github.com/kkroening/ffmpeg-python)

```
pip install ffmpeg-python
```

이미지 프레임을 추출해서 파일로 저장 (참고: OpenCV로도 할 수 있습니다.)

In [2]:
import os
import ffmpeg

#동영상을 이미지로 만들면 파일개수가 많아지니까 별도 폴더를 만들어서 저장하는 게 일반적
if not os.path.isdir("images"):
    os.mkdir("images")

# try:
#     ffmpeg.input("NO_FILE.mp4").output("images/dest%05d.jpg", start_number=0).run(capture_stdout=True, capture_stderr=True)
# except ffmpeg.Error as e:
#     print('stdout:', e.stdout.decode('utf8'))
#     print('stderr:', e.stderr.decode('utf8'))

#ffmpeg은 method chaining이 가능하도록 구현됨.
# ex : dest00001.jpg
ffmpeg.input("virtual_billboard_dest.mp4").output(
    "images/dest%05d.jpg", start_number=0
).run(capture_stdout=True, capture_stderr=True)

ffmpeg.input("virtual_billboard_src.mp4").output(
    "images/src%05d.jpg", start_number=0
).run(capture_stdout=True, capture_stderr=True)

(b'',
 b"ffmpeg version 4.2.2 Copyright (c) 2000-2019 the FFmpeg developers\r\n  built with gcc 9.2.1 (GCC) 20200122\r\n  configuration: --disable-static --enable-shared --enable-gpl --enable-version3 --enable-sdl2 --enable-fontconfig --enable-gnutls --enable-iconv --enable-libass --enable-libdav1d --enable-libbluray --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libopus --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libtheora --enable-libtwolame --enable-libvpx --enable-libwavpack --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libzimg --enable-lzma --enable-zlib --enable-gmp --enable-libvidstab --enable-libvorbis --enable-libvo-amrwbenc --enable-libmysofa --enable-libspeex --enable-libxvid --enable-libaom --enable-libmfx --enable-amf --enable-ffnvcodec --enable-cuvid --enable-d3d11va --enable-nvenc --enable-nvdec --enable-dxva2 --enable-avisynth --enable-libopenm

### Homography 적용

먼저 원본 이미지를 대상 이미지의 어디에 붙일지 픽셀 좌표를 찾습니다.

In [3]:
# 좌표 찾기
import numpy as np
import cv2 as cv


def check_xy(event, x, y, flags, param):

    if event == cv.EVENT_LBUTTONDOWN:
        print(f"[{x},{y}],", end="")


img = cv.imread("images/dest00000.jpg")

cv.namedWindow("Image")  # 창(Window)의 이름 주의
cv.setMouseCallback("Image", check_xy)

while True:

    cv.imshow("Image", img)

    if cv.waitKey(20) & 0xFF == ord("q"):
        break

cv.destroyAllWindows()

[408,198],[491,199],[492,271],[308,242],

[cv::findHomograph()](https://docs.opencv.org/3.4/d9/d0c/group__calib3d.html#ga4abc2ece9fab9398f2e560d53c8c9780)로 원본 이미지를 대상 이미지의 원하는 위치에 붙입니다.

In [4]:
import cv2
import numpy as np


def virtual_billboard(im_src, im_dst):

    pts_src = np.array([[0, 0], [853, 0], [863, 479], [0, 479]])
    pts_dst = np.array([[232, 100], [606, 92], [602, 352], [236, 316]])

    h, status = cv2.findHomography(pts_src, pts_dst)
    # print(h.shape) # (3, 3)

    im_warp = cv2.warpPerspective(im_src, h, (im_dst.shape[1], im_dst.shape[0])) #src를 dest 덮어씌울 부분에 맞게 형상 변경

    cv2.fillConvexPoly(im_dst, pts_dst.astype(int), (0, 0, 0), cv2.LINE_AA) #dest에서 덮어씌울 부분 검은색으로 만들기

    return im_warp + im_dst


im_src = cv2.imread("images/src00000.jpg")
im_dst = cv2.imread("images/dest00000.jpg")

im_comp = virtual_billboard(im_src, im_dst)

cv2.imshow("Warped Source Image", im_comp)

cv2.waitKey(0)
cv2.destroyAllWindows()

### 여러 파일 다루기

폴더 안의 파일 목록 보기

In [5]:
import os

# 지정해준 폴더 안의 모든 파일의 이름을 리스트로
print(os.listdir("images"))

# for루프에서 사용 가능
# for filename in os.listdir("images"):
#     print(filename, end=" ")

['comp00000.jpg', 'comp00001.jpg', 'comp00002.jpg', 'comp00003.jpg', 'comp00004.jpg', 'comp00005.jpg', 'comp00006.jpg', 'comp00007.jpg', 'comp00008.jpg', 'comp00009.jpg', 'comp00010.jpg', 'comp00011.jpg', 'comp00012.jpg', 'comp00013.jpg', 'comp00014.jpg', 'comp00015.jpg', 'comp00016.jpg', 'comp00017.jpg', 'comp00018.jpg', 'comp00019.jpg', 'comp00020.jpg', 'comp00021.jpg', 'comp00022.jpg', 'comp00023.jpg', 'comp00024.jpg', 'comp00025.jpg', 'comp00026.jpg', 'comp00027.jpg', 'comp00028.jpg', 'comp00029.jpg', 'comp00030.jpg', 'comp00031.jpg', 'comp00032.jpg', 'comp00033.jpg', 'comp00034.jpg', 'comp00035.jpg', 'comp00036.jpg', 'comp00037.jpg', 'comp00038.jpg', 'comp00039.jpg', 'comp00040.jpg', 'comp00041.jpg', 'comp00042.jpg', 'comp00043.jpg', 'comp00044.jpg', 'comp00045.jpg', 'comp00046.jpg', 'comp00047.jpg', 'comp00048.jpg', 'comp00049.jpg', 'comp00050.jpg', 'comp00051.jpg', 'comp00052.jpg', 'comp00053.jpg', 'comp00054.jpg', 'comp00055.jpg', 'comp00056.jpg', 'comp00057.jpg', 'comp00058.jp

In [6]:
import glob

# 이름이 조건을 만족하는 파일들
print(len(glob.glob("images/src*.jpg")))
print(len(glob.glob("images/dest*.jpg")))

98
111


### 여러 이미지 파일 변환하기

In [7]:
import glob

file_num = min(len(glob.glob("images/dest*.jpg")), len(glob.glob("images/src*.jpg")))

for i in range(file_num):
    im_src = cv2.imread("images/src" + str(i).zfill(5) + ".jpg") #str(16).zfill(5) == "00016"
    im_dest = cv2.imread("images/dest" + str(i).zfill(5) + ".jpg")

    im_comp = virtual_billboard(im_src, im_dest)

    cv2.imwrite("images/comp" + str(i).zfill(5) + ".jpg", im_comp)

### 이미지 시퀀스를 동영상으로 묶기

In [8]:
import ffmpeg

# 이미지 시퀀스를 동영상으로 묶기
ffmpeg.input("images/comp%05d.jpg").output(
    "virtual_billboard_comp.mp4", start_number=0
).run(capture_stdout=True, capture_stderr=True)

### 동영상 파일에 오디오 추가

동영상에서 오디오 추출

In [1]:
import ffmpeg

ffmpeg.input("virtual_billboard_dest.mp4").output("virtual_billboard_dest.wav").run()

(None, None)

비디오에 오디오 합치기

In [2]:
import ffmpeg

video = ffmpeg.input("virtual_billboard_comp.mp4").video  # get only video channel
audio = ffmpeg.input("virtual_billboard_dest.wav").audio  # get only audio channel
output = ffmpeg.output(
    video,
    audio,
    "virtual_billboard_comp_audio.mp4",
    vcodec="copy",
    acodec="aac",
    strict="experimental",
)

ffmpeg.run(output)

(None, None)